# FHIR Condition Data Quality Profiling

## Purpose

In this notebook, I profile the Silver FHIR Condition dataset and define the
data-quality rules that will later be enforced directly inside the Lakeflow
Bronze-to-Silver transformation.

I am not creating another cleaned Condition table in this notebook.

The purpose of this stage is to:

- identify incomplete or logically invalid Condition records
- distinguish required identifiers from optional FHIR attributes
- validate Condition lifecycle dates
- measure current rule violations
- classify rules as warning, drop, or fail
- prepare reusable Lakeflow expectation definitions

### Source

`health_insurance.silver.fhir_condition`

### Production design

The final production flow will be:

Bronze  
↓  
FHIR Condition transformation  
+  
Lakeflow expectations  
↓  
Validated Silver Condition  
↓  
Gold

### Data-quality approach

FHIR Condition records can legitimately omit some optional attributes such as
an Encounter reference or an abatement date.

I therefore treat missing optional fields as monitoring concerns rather than
automatic rejection conditions.

The most important fields for relational integrity are the Condition ID,
Patient ID, and Condition code.

In [0]:
# loading the current Silver Condition dataset for quality profiling.

from pyspark.sql import functions as F

CONDITION_TABLE = "health_insurance.silver.fhir_condition"

condition_df = spark.table(CONDITION_TABLE)

print(f"Condition rows: {condition_df.count():,}")

condition_df.printSchema()

display(condition_df.limit(10))

In [0]:
# defining candidate Condition quality rules by severity.

CONDITION_WARN_RULES = {
    "encounter_reference_present":
        "encounter_id IS NOT NULL",

    "condition_name_present":
        "condition_name IS NOT NULL",

    "clinical_status_present":
        "clinical_status IS NOT NULL",

    "verification_status_present":
        "verification_status IS NOT NULL",

    "onset_datetime_present":
        "onset_datetime IS NOT NULL"
}


CONDITION_DROP_RULES = {
    "condition_id_present":
        "condition_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "condition_code_present":
        "condition_code IS NOT NULL"
}


CONDITION_FAIL_RULES = {
    "condition_timeline_valid":
        """
        onset_datetime IS NULL
        OR abatement_datetime IS NULL
        OR abatement_datetime >= onset_datetime
        """
}

In [0]:
# measuring how many Condition rows violate each candidate quality rule.

def profile_rules(df, rules, severity):

    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(
                f"NOT ({condition}) OR ({condition}) IS NULL"
            )
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition.strip(),
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
# profiling all proposed Condition quality rules.

condition_quality_results = []

condition_quality_results += profile_rules(
    condition_df,
    CONDITION_WARN_RULES,
    "WARN"
)

condition_quality_results += profile_rules(
    condition_df,
    CONDITION_DROP_RULES,
    "DROP"
)

condition_quality_results += profile_rules(
    condition_df,
    CONDITION_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the Condition quality profile as a structured result.

condition_quality_profile_df = spark.createDataFrame(
    condition_quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    condition_quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# inspecting Condition timeline anomalies directly.

condition_df.select(
    F.count("*").alias("total_conditions"),

    F.sum(
        F.col("onset_datetime").isNull().cast("int")
    ).alias("missing_onset"),

    F.sum(
        F.col("abatement_datetime").isNull().cast("int")
    ).alias("missing_abatement"),

    F.sum(
        (
            F.col("abatement_datetime")
            < F.col("onset_datetime")
        ).cast("int")
    ).alias("abatement_before_onset")
).show()

In [0]:
# inspecting the actual Condition status domains
# before finalizing controlled-value quality rules.

display(
    condition_df
    .groupBy(
        "clinical_status",
        "verification_status"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

In [0]:
# defining the finalized Condition quality contract.

CONDITION_WARN_RULES = {
    "encounter_reference_present":
        "encounter_id IS NOT NULL",

    "condition_name_present":
        "condition_name IS NOT NULL",

    "recognized_clinical_status":
        "clinical_status IN ('ACTIVE', 'RESOLVED')",

    "recognized_verification_status":
        "verification_status IN ('CONFIRMED')",

    "onset_datetime_present":
        "onset_datetime IS NOT NULL"
}


CONDITION_DROP_RULES = {
    "condition_id_present":
        "condition_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "condition_code_present":
        "condition_code IS NOT NULL"
}


CONDITION_FAIL_RULES = {
    "condition_timeline_valid":
        """
        onset_datetime IS NULL
        OR abatement_datetime IS NULL
        OR abatement_datetime >= onset_datetime
        """
}

## Publish approved Condition quality rules

I have completed the Condition quality profiling and finalized the approved
WARN, DROP, and FAIL rules.

I now publish these rules directly into the central Unity Catalog governance
table so the Lakeflow pipeline can retrieve them dynamically.

This keeps the published Condition quality contract connected to the notebook
where I developed and validated it.

In [0]:
# converting the finalized Condition quality contract
# into rows for the central governance repository.

from pyspark.sql import functions as F

def build_rule_rows(dataset, severity, rules, description, source_notebook):

    return [
        (
            dataset,
            rule_name,
            constraint.strip(),
            severity,
            True,
            description,
            source_notebook,
            "data_engineering",
            1
        )
        for rule_name, constraint in rules.items()
    ]

In [0]:
# preparing all approved Condition rules for publication.

condition_rule_rows = []

condition_rule_rows += build_rule_rows(
    "condition",
    "WARN",
    CONDITION_WARN_RULES,
    "FHIR Condition quality monitoring rule",
    "04-data-quality/03_condition_quality_profile"
)

condition_rule_rows += build_rule_rows(
    "condition",
    "DROP",
    CONDITION_DROP_RULES,
    "FHIR Condition record validity rule",
    "04-data-quality/03_condition_quality_profile"
)

condition_rule_rows += build_rule_rows(
    "condition",
    "FAIL",
    CONDITION_FAIL_RULES,
    "Critical FHIR Condition pipeline rule",
    "04-data-quality/03_condition_quality_profile"
)

In [0]:
# creating the Condition quality-rule publication DataFrame.

condition_rules_df = (
    spark.createDataFrame(
        condition_rule_rows,
        [
            "dataset",
            "rule_name",
            "constraint",
            "severity",
            "is_active",
            "description",
            "source_notebook",
            "owner",
            "version"
        ]
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(condition_rules_df)

In [0]:
# exposing the finalized Condition rules
# as a temporary view for idempotent publishing.

condition_rules_df.createOrReplaceTempView(
    "condition_quality_rule_updates"
)

In [0]:
%sql
-- publishing the approved Condition rules
-- directly from this profiling notebook.

MERGE INTO health_insurance.governance.quality_rules AS target

USING condition_quality_rule_updates AS source

ON target.dataset = source.dataset
AND target.rule_name = source.rule_name

WHEN MATCHED THEN UPDATE SET

    target.constraint = source.constraint,
    target.severity = source.severity,
    target.is_active = source.is_active,
    target.description = source.description,
    target.source_notebook = source.source_notebook,
    target.owner = source.owner,

    target.version =
        CASE
            WHEN target.constraint <> source.constraint
              OR target.severity <> source.severity
            THEN COALESCE(target.version, 1) + 1
            ELSE target.version
        END,

    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN INSERT (
    dataset,
    rule_name,
    constraint,
    severity,
    is_active,
    description,
    source_notebook,
    owner,
    version,
    created_at,
    updated_at
)

VALUES (
    source.dataset,
    source.rule_name,
    source.constraint,
    source.severity,
    source.is_active,
    source.description,
    source.source_notebook,
    source.owner,
    source.version,
    source.created_at,
    source.updated_at
);

In [0]:
%sql
-- verifying the Condition rules published by this notebook.

SELECT
    dataset,
    rule_name,
    severity,
    constraint,
    source_notebook,
    version,
    is_active,
    updated_at
FROM health_insurance.governance.quality_rules
WHERE dataset = 'condition'
ORDER BY severity, rule_name;